For this warmup, you've been provided two datasets, the first contains gage heights of the West Fork of the Stones River near Murfreesboro (https://waterdata.usgs.gov/monitoring-location/03428200/#parameterCode=00065&showMedian=false&startDT=2023-11-22&endDT=2024-11-22), recorded at 11:00 PM. and the second give daily rainfall totals, also for the West Fork of the Stones River near Murfreesboro (https://www.tva.com/environment/lake-levels/rainfall-gauge-data).

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import datetime

In [2]:
river = pd.read_csv("../data/stones_river.csv")
river.head()

,datetime,date,gage_height_ft
0,2008-05-28 23:00:00,2008-05-28,2.67
1,2008-05-29 23:00:00,2008-05-29,2.61
2,2008-05-30 23:00:00,2008-05-30,2.51
3,2008-05-31 23:00:00,2008-05-31,2.51
4,2008-06-01 23:00:00,2008-06-01,3.76


In [3]:
rainfall = pd.read_csv("../data/rainfall.csv")
rainfall.head()

,date,rainfall_in
0,2008-01-01,0.0
1,2008-01-02,0.0
2,2008-01-03,0.0
3,2008-01-04,0.0
4,2008-01-05,0.0


**Part 1:** Merge the two datasets, making sure to keep all records in both.

In [4]:
river_rain_df = pd.merge(left=river, right=rainfall, on="date", how="outer").sort_values("date")

In [5]:
river_rain_df["date"] = pd.to_datetime(river_rain_df["date"], format = "%Y-%m-%d")

**Part 2:** Fit a linear regression model with target being gage_height_ft and predictor being rainfall_in.

How well does this model do? What does the model estimate for the average gage height on a day with 1.5 inches of rainfall?

In [6]:
rain_linreg = smf.ols("gage_height_ft ~ rainfall_in", data=river_rain_df).fit()

In [7]:
rain_linreg.params

Intercept      2.801497
rainfall_in    1.404694
dtype: float64

In [8]:
rain_linreg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         gage_height_ft   R-squared:                       0.232
Model:                            OLS   Adj. R-squared:                  0.232
Method:                 Least Squares   F-statistic:                     1795.
Date:                Mon, 08 Dec 2025   Prob (F-statistic):               0.00
Time:                        20:10:57   Log-Likelihood:                -8821.6
No. Observations:                5950   AIC:                         1.765e+04
Df Residuals:                    5948   BIC:                         1.766e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       2.8015      0.015    188.769      0.000       2.772       2.831
rainfall_in     1.4047      0.033     42.362      0.000       1.340       1.470
==============================================================================
Omnibus:                     5843.819   Durbin-Watson:                   1.202
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           474184.184
Skew:                           4.626   Prob(JB):                         0.00
Kurtosis:                      45.744   Cond. No.                         2.48
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

gage_height_ft is equal to the intercept of 2.801497 + 1.404694 * rainfall_in

gage_height_ft is equal to the intercept of 2.801497 + 1.404694 * 1.5 rainfall_in = 4.908538 gage height.

In [9]:
mean_le = river_rain_df.groupby("gage_height_ft")["rainfall_in"].mean().reset_index()
mean_le["mean_estimate"] = rain_linreg.predict(mean_le)
mean_le

,gage_height_ft,rainfall_in,mean_estimate
0,1.66,0.00,2.801497
1,1.73,0.00,2.801497
2,1.75,0.00,2.801497
3,1.77,0.24,3.138624
4,1.79,0.00,2.801497
...,...,...,...
485,16.64,2.20,5.891824
486,17.78,1.41,4.782116
487,18.21,0.31,3.236953
488,18.82,3.25,7.366752


In [10]:
#mean_le.plot(kind='line', x='gage_height_ft', y='rainfall_in', color='green')
#plt.plot(mean_le['rainfall_in'], rain_linreg.fittedvalues, color='orange')
#plt.ylabel('rainfall_ins')
#plt.legend(['rainfall_in', 'Prediction']);

**Part 3:** Create a new column which contains, for each row, the total rainfall over the last 7 days.

Then fit a new model that uses this column as the predictor. Does this model do better of worse than the one with just the daily rainfall total?

river_rain_df was sorted by date above

In [11]:
river_rain_df["rainfall_over_last_7_days"] = river_rain_df.rolling(window="7D", on="date")["rainfall_in"].sum()
river_rain_df

,datetime,date,gage_height_ft,rainfall_in,rainfall_over_last_7_days
0,NaN,2008-01-01,NaN,0.00,0.00
1,NaN,2008-01-02,NaN,0.00,0.00
2,NaN,2008-01-03,NaN,0.00,0.00
3,NaN,2008-01-04,NaN,0.00,0.00
4,NaN,2008-01-05,NaN,0.00,0.00
...,...,...,...,...,...
6165,2024-11-17 23:00:00,2024-11-17,2.37,0.00,0.73
6166,2024-11-18 23:00:00,2024-11-18,2.26,0.00,0.47
6167,2024-11-19 23:00:00,2024-11-19,2.64,0.05,0.52
6168,2024-11-20 23:00:00,2024-11-20,2.49,0.60,1.12


In [12]:
rain_7_linreg = smf.ols("gage_height_ft ~ rainfall_over_last_7_days", data=river_rain_df).fit()

In [13]:
rain_7_linreg.params

Intercept                    2.547956
rainfall_over_last_7_days    0.422712
dtype: float64

In [14]:
rain_7_linreg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         gage_height_ft   R-squared:                       0.180
Model:                            OLS   Adj. R-squared:                  0.180
Method:                 Least Squares   F-statistic:                     1303.
Date:                Mon, 08 Dec 2025   Prob (F-statistic):          3.50e-258
Time:                        20:10:57   Log-Likelihood:                -9016.8
No. Observations:                5950   AIC:                         1.804e+04
Df Residuals:                    5948   BIC:                         1.805e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     2.5480      0.020    130.227      0.000       2.510       2.586
rainfall_over_last_7_days     0.4227      0.012     36.097      0.000       0.400       0.446
==============================================================================
Omnibus:                     6589.377   Durbin-Watson:                   1.131
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           658385.003
Skew:                           5.636   Prob(JB):                         0.00
Kurtosis:                      53.286   Cond. No.                         2.75
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

The P values are both 0 signify that they are both statistically significant. the Rsquared value for the rolling 7 day has a lower value and thus is not as good of a fit as the original model  (perfect match is rsquared of 1) 

In [15]:
2.801497 + (0.422712 * 1.5)

3.435565

gage_height_ft is equal to the intercept of 2.801497 + 0.422712 * 1.5 rainfall_in over a rolling 7 day period =  <br> 3.435565 gage height.

## Stretch Questions

**Part 4:** Fit a model which uses the gage height from the day before to estimate the average gage height for a given day. Hint: You may want to make use of the [shift method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html) to create a column containing yesterday's gage height.

What does the model estimate will be the average gage height tomorrow if the gage height was 3 feet today? 

What does the model estimate will be the average gage height tomorrow if the gage height was 5 feet today? 

In [16]:
river_rain_df["yesterdays_gage_height_ft"] = river_rain_df["gage_height_ft"].shift(periods=-1)#, freq="D")

In [17]:
river_rain_df

,datetime,date,gage_height_ft,rainfall_in,rainfall_over_last_7_days,yesterdays_gage_height_ft
0,NaN,2008-01-01,NaN,0.00,0.00,NaN
1,NaN,2008-01-02,NaN,0.00,0.00,NaN
2,NaN,2008-01-03,NaN,0.00,0.00,NaN
3,NaN,2008-01-04,NaN,0.00,0.00,NaN
4,NaN,2008-01-05,NaN,0.00,0.00,NaN
...,...,...,...,...,...,...
6165,2024-11-17 23:00:00,2024-11-17,2.37,0.00,0.73,2.26
6166,2024-11-18 23:00:00,2024-11-18,2.26,0.00,0.47,2.64
6167,2024-11-19 23:00:00,2024-11-19,2.64,0.05,0.52,2.49
6168,2024-11-20 23:00:00,2024-11-20,2.49,0.60,1.12,2.45


In [18]:
rain_yesterday_linreg = smf.ols("gage_height_ft ~ yesterdays_gage_height_ft", data=river_rain_df).fit()

In [19]:
rain_yesterday_linreg.params

Intercept                    1.427773
yesterdays_gage_height_ft    0.528028
dtype: float64

In [20]:
1.427773 + (0.528028 * 3)

3.011857

In [21]:
rain_yesterday_linreg.params.Intercept + (rain_yesterday_linreg.params.yesterdays_gage_height_ft * 5)

np.float64(4.067913978339341)

gage_height_ft is equal to the intercept of 1.427773 + 0.528028 * 3 rainfall_in = 3.011857 gage height.

In [22]:
1.427773 + (0.528028 * 5)

4.067913

In [23]:
rain_yesterday_linreg.params.Intercept + (rain_yesterday_linreg.params.yesterdays_gage_height_ft * 3)

np.float64(3.011857668806147)

gage_height_ft is equal to the intercept of 1.427773 + 0.528028 * 3 rainfall_in = 4.067913 gage height.

In [24]:
rain_yesterday_linreg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         gage_height_ft   R-squared:                       0.281
Model:                            OLS   Adj. R-squared:                  0.281
Method:                 Least Squares   F-statistic:                     2323.
Date:                Mon, 08 Dec 2025   Prob (F-statistic):               0.00
Time:                        20:10:57   Log-Likelihood:                -8575.3
No. Observations:                5932   AIC:                         1.715e+04
Df Residuals:                    5930   BIC:                         1.717e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     1.4278      0.036     39.901      0.000       1.358       1.498
yesterdays_gage_height_ft     0.5280      0.011     48.200      0.000       0.507       0.550
==============================================================================
Omnibus:                     6106.443   Durbin-Watson:                   2.130
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           725669.013
Skew:                           4.862   Prob(JB):                         0.00
Kurtosis:                      56.305   Cond. No.                         9.48
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Part 5:** Fit a model which uses the gage height from the day before and the rainfall total to estimate the average gage height for a given day.

What does the model estimate will be the average gage height if the gage height tomorrow was 5 feet today and it didn't rain today? 

What does the model estimate will be the average gage height if the gage height tomorrow was 5 feet today and it rained 2 inches today? 

In [25]:
#river_rain_df[""] = river_rain_df["rainfall_in"].shift(periods=-1)#, freq="D")

In [26]:
rain_yest_rain_linreg = smf.ols("gage_height_ft ~ yesterdays_gage_height_ft + rainfall_in", data=river_rain_df).fit()

In [27]:
rain_yest_rain_linreg.params

Intercept                    1.504504
yesterdays_gage_height_ft    0.442724
rainfall_in                  1.114026
dtype: float64

In [28]:
rain_yest_rain_linreg.params.Intercept + rain_yest_rain_linreg.params.yesterdays_gage_height_ft * 5 + rain_yest_rain_linreg.params.rainfall_in * 0

np.float64(3.7181251573282355)

In [29]:
rain_yest_rain_linreg.params.Intercept + rain_yest_rain_linreg.params.yesterdays_gage_height_ft * 5 + rain_yest_rain_linreg.params.rainfall_in * 2

np.float64(5.946177337441986)

In [30]:
rain_yest_rain_linreg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         gage_height_ft   R-squared:                       0.421
Model:                            OLS   Adj. R-squared:                  0.421
Method:                 Least Squares   F-statistic:                     2157.
Date:                Mon, 08 Dec 2025   Prob (F-statistic):               0.00
Time:                        20:10:57   Log-Likelihood:                -7934.3
No. Observations:                5932   AIC:                         1.587e+04
Df Residuals:                    5929   BIC:                         1.589e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     1.5045      0.032     46.746      0.000       1.441       1.568
yesterdays_gage_height_ft     0.4427      0.010     43.882      0.000       0.423       0.463
rainfall_in                   1.1140      0.029     37.820      0.000       1.056       1.172
==============================================================================
Omnibus:                     5537.598   Durbin-Watson:                   2.177
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           578231.652
Skew:                           4.134   Prob(JB):                         0.00
Kurtosis:                      50.656   Cond. No.                         9.61
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""